# Chapter 4 — Calibration, conformal prediction, selective prediction

Companion notebook for *Evaluating Large Language Models in High-Stakes Domains* (Costa, 2026). All data in this notebook are simulated; the generating process is in the code and described in the text.

In [ ]:
"""Figures and worked-example numbers for 'Evaluating Large Language Models in High-Stakes Domains'.
All data are analytic or simulated; generating processes are stated in the text.
Run:  python figures.py   (writes PNGs and results.json into ./figures)
"""
import os, json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize, minimize_scalar

OUT = os.path.join(os.getcwd(), "figures_out")
os.makedirs(OUT, exist_ok=True)

# palette (categorical, fixed order) and ink
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2,
    "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "legend.frameon": False, "figure.dpi": 200, "savefig.dpi": 200,
    "savefig.facecolor": "white",
})
results = {}
se = lambda x: x.std(ddof=1)/np.sqrt(len(x))
def wilson(x, n, z=1.96):
    p = x/n; c = (p + z*z/(2*n))/(1 + z*z/n)
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/(1 + z*z/n)
    return c-h, c+h
def clopper_upper(x, n, conf=0.95):
    return 1.0 if x == n else stats.beta.ppf(conf, x+1, n-x)


## Figure 2: reliability diagram (Platt scaling)

In [ ]:
rng = np.random.default_rng(11)
N = 4000
true_p = rng.beta(5, 2, N)
correct = rng.binomial(1, true_p)
conf = np.clip(0.55 + 0.6*(true_p-0.5) + rng.normal(0, 0.08, N), 0.05, 0.99)
conf = np.minimum(1.0, conf*1.18)
conf = np.round(conf*20)/20
bins = np.linspace(0, 1, 11)
def reliability(cf, y):
    w = np.clip(np.digitize(cf, bins)-1, 0, 9)
    acc = np.array([y[w==b].mean() if (w==b).any() else np.nan for b in range(10)])
    cfm = np.array([cf[w==b].mean() if (w==b).any() else np.nan for b in range(10)])
    cnt = np.array([(w==b).sum() for b in range(10)])
    return acc, cfm, cnt, np.nansum(cnt/len(y)*np.abs(acc-cfm))
acc_b, conf_b, cnt_b, ece = reliability(conf, correct)
logit = lambda p: np.log(p/(1-p))
half = rng.permutation(N); fit, ev = half[:N//2], half[N//2:]
zf = logit(np.clip(conf[fit], 1e-3, 1-1e-3))
def nll(ab):
    qf = 1/(1+np.exp(-(ab[0]*zf + ab[1])))
    return -np.mean(correct[fit]*np.log(qf)+(1-correct[fit])*np.log(1-qf))
a_pl, b_pl = minimize(nll, x0=[1.0, 0.0]).x
q_ev = 1/(1+np.exp(-(a_pl*logit(np.clip(conf[ev],1e-3,1-1e-3)) + b_pl)))
acc2, conf2, cnt2, ece2 = reliability(q_ev, correct[ev])
results["ch4"] = {"ECE_RAW": f"{ece:.3f}", "ECE_PL": f"{ece2:.3f}", "PLATT_A": f"{a_pl:.2f}", "PLATT_B": f"{b_pl:+.2f}",
                  "BRIER_RAW": f"{np.mean((conf-correct)**2):.3f}", "BRIER_PL": f"{np.mean((q_ev-correct[ev])**2):.3f}"}
fig, ax = plt.subplots(1, 2, figsize=(7.2, 3.0), gridspec_kw={"width_ratios": [1.1, 1]})
ax[0].plot([0,1],[0,1], ls="--", color=INK2, lw=1, label="perfect calibration")
ok1 = cnt_b >= 30; ok2 = cnt2 >= 30
ax[0].plot(conf_b[ok1], acc_b[ok1], "o-", color=C1, lw=2, ms=5, label=f"raw confidence (ECE {ece:.3f})")
ax[0].plot(conf2[ok2], acc2[ok2], "s-", color=C2, lw=2, ms=5, label=f"Platt-scaled (ECE {ece2:.3f})")
ax[0].set_xlabel("Stated confidence (bin mean)"); ax[0].set_ylabel("Observed accuracy")
ax[0].set_xlim(0,1); ax[0].set_ylim(0,1); ax[0].set_aspect("equal")
ax[0].set_title("(a) Reliability diagram", loc="left", fontsize=9, color=INK)
ax[0].legend(loc="upper left", fontsize=7.5)
ax[1].bar(bins[:-1]+0.05, cnt_b/N, width=0.09, color=C1, alpha=0.9, label="raw")
ax[1].bar(bins[:-1]+0.05, cnt2/len(ev), width=0.09, color=C2, alpha=0.55, label="Platt-scaled")
ax[1].set_xlabel("Stated confidence"); ax[1].set_ylabel("Fraction of items")
ax[1].set_title("(b) Confidence histogram", loc="left", fontsize=9, color=INK)
ax[1].legend(loc="upper left")
fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig2_reliability.png")); plt.close(fig)

## Figure 3: conformal coverage

In [ ]:
rng = np.random.default_rng(3)
alpha = 0.10; n_cal = 200; n_test = 20000; Rr = 3000
k_ord = int(np.ceil((n_cal+1)*(1-alpha)))
K = 5
def gen(m):
    y = rng.integers(0, K, m)
    logits = rng.normal(0, 1.0, (m, K)); logits[np.arange(m), y] += rng.gamma(2.0, 1.2, m)
    p = np.exp(logits); p /= p.sum(1, keepdims=True)
    return y, p
cov = np.empty(Rr); size = np.empty(Rr)
for r in range(Rr):
    yc, pc = gen(n_cal); yt, pt = gen(n_test)
    scores = 1 - pc[np.arange(n_cal), yc]
    qhat = np.sort(scores)[k_ord-1]
    sets = (1 - pt) <= qhat
    cov[r] = sets[np.arange(n_test), yt].mean(); size[r] = sets.sum(1).mean()
beta = stats.beta(k_ord, n_cal+1-k_ord)
results["ch4"].update({"COV_MEAN": f"{cov.mean():.4f}", "COV_SD": f"{cov.std():.4f}",
                       "SET_MEAN": f"{size.mean():.2f}", "UPPER": f"{1-alpha+1/(n_cal+1):.4f}"})
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.9))
ax[0].hist(cov, bins=40, density=True, color=C1, alpha=0.85, label="empirical (3,000 calibration draws)")
xs = np.linspace(0.82, 0.97, 400); ax[0].plot(xs, beta.pdf(xs), color=C2, lw=2, label="Beta(k, n+1−k) theory")
ax[0].axvline(1-alpha, color=INK2, ls="--", lw=1); ax[0].text(1-alpha+0.0015, 1.0, "1−α", color=INK2, fontsize=8)
ax[0].set_xlabel("Test coverage"); ax[0].set_ylabel("Density")
ax[0].set_title("(a) Coverage over calibration draws, n = 200", loc="left", fontsize=9, color=INK)
ax[0].legend(fontsize=7.5, loc="upper left")
yc, pc = gen(n_cal); yt, pt = gen(n_test); scores = 1 - pc[np.arange(n_cal), yc]
levels = np.linspace(0.5, 0.99, 50); sz=[]
for a in 1-levels:
    kk = min(n_cal, int(np.ceil((n_cal+1)*(1-a)))); qq = np.sort(scores)[kk-1]
    sz.append(((1-pt) <= qq).sum(1).mean())
ax[1].plot(levels, sz, color=C1, lw=2)
ax[1].set_xlabel("Target coverage 1−α"); ax[1].set_ylabel("Average set size (of 5 options)")
ax[1].set_title("(b) Price of coverage: set size", loc="left", fontsize=9, color=INK)
ax[1].axvline(0.9, color=INK2, ls="--", lw=1)
fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig3_conformal.png")); plt.close(fig)

## Figure 4: risk-coverage curves

In [ ]:
rng = np.random.default_rng(5)
N = 3000
true_p = rng.beta(4, 1.5, N); correct = rng.binomial(1, true_p)
verbal = np.clip(true_p + rng.normal(0, 0.30, N), 0, 1)
consist = np.clip(true_p + rng.normal(0, 0.12, N), 0, 1)
def rc_curve(cf):
    order = np.argsort(-cf); err = 1-correct[order]
    cv = np.arange(1, N+1)/N; risk = np.cumsum(err)/np.arange(1, N+1)
    return cv, risk, np.trapezoid(risk, cv)
START = 30
cv1, r1, aurc1 = rc_curve(verbal); cv2, r2, aurc2 = rc_curve(consist)
rand_risk = 1-correct.mean()
results["ch4"].update({"AURC_V": f"{aurc1:.3f}", "AURC_C": f"{aurc2:.3f}", "BASE_RISK": f"{rand_risk:.3f}",
                       "R70_V": f"{r1[int(0.7*N)-1]:.3f}", "R70_C": f"{r2[int(0.7*N)-1]:.3f}"})
fig, ax = plt.subplots(figsize=(4.6, 3.0))
ax.plot(cv1[START:], r1[START:], color=C1, lw=2, label=f"verbalized confidence (AURC {aurc1:.3f})")
ax.plot(cv2[START:], r2[START:], color=C2, lw=2, label=f"consistency-based (AURC {aurc2:.3f})")
ax.axhline(rand_risk, color=INK2, ls="--", lw=1); ax.text(0.50, rand_risk+0.006, "answer everything", color=INK2, fontsize=8)
ax.axhline(0.05, color=C3, ls=":", lw=1.2); ax.text(0.02, 0.053, "tolerated risk 5%", color=C3, fontsize=8)
ax.set_xlabel("Coverage (fraction of items answered)"); ax.set_ylabel("Selective risk (error rate on answered items)")
ax.set_xlim(0,1); ax.set_ylim(0, 0.37); ax.legend(loc="upper left", fontsize=7.5)
ax.set_title("Risk–coverage curves for two confidence signals", loc="left", fontsize=9, color=INK)
fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig4_risk_coverage.png")); plt.close(fig)

In [ ]:
import json
print(json.dumps(results, indent=2, ensure_ascii=False))